# Day 3 · 3교시 [주석본] ML 파이프라인 — 한 줄씩 뜯어보기

이 노트북은 `03_ml_pipeline.ipynb` 와 **똑같은 코드**에 **설명과 줄별 주석**을 붙인 것이다.
**딥러닝은 해봤지만 고전 ML(scikit-learn)은 낯선 사람**을 기준으로 썼다.

---

## 0. 먼저 — 우리가 지금 뭘 하려는 건가?

**와인 178병의 화학 성분을 재서, 그 와인이 어느 품종인지 맞히는 모델을 만든다.**

이탈리아 같은 지역에서 자란 **세 가지 포도 품종(cultivar)** 으로 만든 와인들이 있다.
각 와인병마다 화학 실험실에서 **13가지 성분을 측정**해 놓았다 — 알코올 도수, 사과산,
마그네슘, 색의 진하기, 프롤린(아미노산) 같은 것들이다.

| | 내용 |
|---|---|
| **입력 X** | 13개 숫자 (화학 성분 측정치) |
| **출력 y** | 0 / 1 / 2 (세 품종 중 하나) |
| **데이터 크기** | **178병** |
| **풀려는 문제** | 다중 클래스 분류 (multi-class classification) |

### 딥러닝 하던 사람이 여기서 가장 놀라는 것 3가지

**① 데이터가 178개다.** 오타가 아니다. 이미지넷 128만 장, 토큰 수십억 개를 보다가
오면 "이게 데이터셋이라고?" 싶다. **그래서 딥러닝을 안 쓴다.** 파라미터 수백만 개짜리
신경망을 178개 샘플로 학습시키면 그냥 통째로 외워 버린다. 이런 **작은 표(tabular)
데이터**에서는 단순한 모델이 더 잘하고, 더 빠르고, 왜 그렇게 판단했는지 설명도 된다.

**② 특성(feature)을 모델이 안 만든다.** 딥러닝은 raw 픽셀·raw 텍스트를 넣으면 중간층이
알아서 특징을 뽑아낸다. 여기서는 **사람(화학자)이 이미 뽑아 놓았다** — 13개 측정치가
곧 특성이다. 고전 ML 에서 "feature engineering 이 실력"이라는 말이 이래서 나온다.

**③ 학습 루프가 안 보인다.** `for epoch in range(...)`, `loss.backward()`,
`optimizer.step()` 이 없다. `.fit(X, y)` **한 줄이 그 전부를 대신한다.** sklearn 이
안에서 알아서 수렴할 때까지 돌린다.

### 오늘 이 데이터로 하려는 진짜 목적

와인 분류 자체가 목적이 아니다. **"AI가 짜 준 ML 코드가 조용히 틀리는 방식"** 을 보는 게
목적이다(교안 3.8). 와인 데이터는 그걸 보여주기 위한 **작고 안전한 실험대**일 뿐이다.

> 의존성: `pip install scikit-learn numpy` · 전부 오프라인·시드 고정

## 1. 데이터 로드·탐색

**탐색의 목적은 구경이 아니라 "전처리에서 뭘 해야 하는지" 정하는 것이다.**
딥러닝에서 데이터셋 통계를 먼저 확인하는 것과 같은 습관이다 —
이미지라면 채널별 mean/std 를 보고, 표 데이터라면 아래 셋을 본다:

1. **결측치**가 있나? (있으면 채우거나 버려야 한다)
2. **스케일**이 특성마다 얼마나 다른가? (다르면 표준화 필요)
3. **클래스 균형**은 맞나? (한쪽으로 쏠리면 정확도가 거짓말을 한다 → 7절)

In [ ]:
import numpy as np                          # 수치 계산 (배열·통계)
from sklearn.datasets import load_wine       # sklearn 에 내장된 예제 데이터 — 다운로드 불필요

wine = load_wine()                           # Bunch 객체(딕셔너리 비슷)로 로드
X = wine.data                                # 입력: (178, 13) 실수 배열 — 178병 × 13성분
y = wine.target                              # 정답: (178,) 정수 배열 — 각 원소가 0/1/2
names = [str(n) for n in wine.target_names]  # 클래스 이름 3개. str() 은 numpy 표기 제거용

# X.shape 는 딥러닝의 (batch, features) 와 같은 의미. 다만 여기선 batch 가 아니라 '전체'다.
print("데이터:", X.shape, "| 클래스:", names)

# np.bincount(y): y 안에서 0,1,2 가 각각 몇 번 나오는지 센다 → 클래스 분포
print("클래스 분포:", {n: int(c) for n, c in zip(names, np.bincount(y))})

# np.isnan(X): 결측(NaN) 위치를 True 로 표시 → .sum() 으로 개수를 센다
print("결측치:", int(np.isnan(X).sum()))

# 특성마다 단위가 달라 값의 범위도 제각각이다. 이 격차가 표준화가 필요한 이유다.
print(f"특성 값 범위: {X.min():.2f} ~ {X.max():.1f}   ← 스케일 차이가 크다 → 표준화 필요")

### 방금 결과가 말해 주는 것

```
데이터: (178, 13) | 클래스: ['class_0', 'class_1', 'class_2']
클래스 분포: {'class_0': 59, 'class_1': 71, 'class_2': 48}
결측치: 0
특성 값 범위: 0.13 ~ 1680.0
```

- **결측치 0** → 채워 넣는 작업(imputation) 불필요. 운이 좋다.
- **클래스 59 : 71 : 48** → 크게 치우치지 않았다. (심하게 쏠린 경우는 7절에서 따로 본다)
- **값 범위 0.13 ~ 1680** ← **이게 중요하다.**

마지막 것을 특성별로 뜯어보면 이렇다:

| 특성 | 범위 |
|---|---|
| `proline` (아미노산) | **278 ~ 1680** |
| `magnesium` | 70 ~ 162 |
| `nonflavanoid_phenols` | **0.13 ~ 0.66** |

`proline` 과 `nonflavanoid_phenols` 는 값의 크기가 **2500배** 차이 난다.
그냥 넣으면 모델이 "숫자가 큰 특성이 중요한 특성"이라고 착각한다 —
거리·내적 계산에서 큰 값이 전부 지배해 버리기 때문이다.

> 🧠 **딥러닝 감각으로 번역하면**: 이미지를 0~255 그대로 넣지 않고 `/255` 하거나
> 채널별 mean/std 로 정규화하는 것과 **정확히 같은 이유**다. 입력 스케일을 맞춰 주지
> 않으면 학습이 한쪽으로 쏠린다. 다음 절에서 이걸 한다.

## 2. 전처리 — **분할 먼저**, 스케일은 그 다음

여기가 이 노트북에서 **가장 중요한 순서**다.

### train / test 가 뭔가 (딥러닝의 train/val 과 같다)

가진 178병을 둘로 나눈다:
- **train (124병)**: 모델이 보고 배우는 데이터
- **test (54병)**: 모델이 **한 번도 못 본** 데이터. 실전 성능을 재는 용도.

test 를 "처음 보는 데이터"로 유지하는 게 전부다. 조금이라도 미리 엿보면 점수가 부풀려진다.

### 그래서 순서가 생명이다

```
✓ 올바름:  분할한다  →  train 으로만 평균·표준편차를 구한다  →  그 값으로 둘 다 변환
✗ 틀림:    전체로 평균·표준편차를 구한다  →  분할한다
```

틀린 쪽은 **test 의 분포 정보가 이미 변환식에 섞여 들어간다.** 이걸 **데이터 누수
(data leakage)** 라 부르고, 6절에서 점수가 얼마나 뻥튀기되는지 숫자로 본다.

> 🧠 **딥러닝에서도 똑같은 실수가 있다** — validation set 을 포함해서 정규화 통계를
> 계산하거나, 전체 데이터에 augmentation 통계를 맞추는 경우. 프레임워크만 다를 뿐
> 같은 병이다.

In [ ]:
from sklearn.model_selection import train_test_split   # 데이터를 train/test 로 쪼개는 함수
from sklearn.preprocessing import StandardScaler          # 표준화: (값 - 평균) / 표준편차

# ── 1단계: 먼저 쪼갠다 ────────────────────────────────────────────
Xtr, Xte, ytr, yte = train_test_split(
    X, y,
    test_size=0.3,        # 30% 를 test 로 → 178 × 0.3 ≈ 54병
    random_state=42,      # 난수 시드 고정 → 몇 번 돌려도 같은 분할 (재현성)
    stratify=y,           # 중요: 원래 클래스 비율(59:71:48)을 train/test 양쪽에 그대로 유지
)

# ── 2단계: 그 다음 스케일링 ──────────────────────────────────────
scaler = StandardScaler().fit(Xtr)   # fit = 평균·표준편차를 '계산'한다. train 만 보고!
Xtr_s = scaler.transform(Xtr)        # transform = 계산해 둔 값으로 '변환'만
Xte_s = scaler.transform(Xte)        # test 도 train 기준으로 변환 (여기서 fit 하면 누수!)

print("분할:", Xtr.shape[0], "train /", Xte.shape[0], "test")

### `fit` 과 `transform` 을 구분하는 게 핵심이다

| 메서드 | 하는 일 | 누구를 보고? |
|---|---|---|
| `.fit(Xtr)` | 각 특성의 **평균·표준편차를 계산해 기억** | **train 만** |
| `.transform(X)` | 기억해 둔 값으로 `(x - 평균) / 표준편차` **변환** | 아무 데이터나 |
| `.fit_transform(X)` | 위 둘을 한 번에 | ⚠️ **test 에 쓰면 누수** |

`Xte_s = scaler.fit_transform(Xte)` 라고 쓰면 test 의 평균으로 test 를 변환하게 된다 —
실전에서는 불가능한 일이다(실전에선 새 데이터 한 건이 들어올 때 그 한 건의 평균을 알 수 없다).

> 💡 `stratify=y` 를 넣은 이유: 안 넣으면 우연히 test 에 `class_2` 가 몰리거나 빠질 수 있다.
> 178개처럼 작은 데이터에서는 이 우연이 점수를 크게 흔든다.

## 3. 학습 — 가장 단순한 모델부터 (baseline)

**baseline = 기준선.** 복잡한 걸 시도하기 전에 "제일 단순한 방법으로 하면 몇 점?"을
먼저 재 둔다. 이게 있어야 나중에 복잡한 모델을 만들었을 때 **"정말 나아진 건가?"** 를
판단할 수 있다.

### 로지스틱 회귀 = 은닉층 없는 신경망

이름에 "회귀"가 들어가지만 **분류 모델**이다. 딥러닝 용어로 정확히 번역하면:

```
Linear(13 → 3)  →  softmax        # 이게 전부. 은닉층 0개.
```

즉 **가장 얕은 신경망**이다. 파라미터는 13×3 + 3 = 42개뿐이다.
(비교: ResNet-50 은 2500만 개)

### `.fit()` 안에서 무슨 일이 일어나나

딥러닝이라면 이렇게 썼을 것이다:
```python
for epoch in range(100):
    loss = criterion(model(X), y)
    loss.backward()
    optimizer.step()
```
sklearn 은 이걸 `.fit(X, y)` **한 줄로 감춘다.** 내부적으로는 똑같이 손실을 줄이는
최적화를 돈다(`max_iter=1000` 이 그 반복 상한이다). 감춰져 있을 뿐 없는 게 아니다.

In [ ]:
from sklearn.linear_model import LogisticRegression   # 선형 분류기 (= 은닉층 0개 신경망)
from sklearn.metrics import accuracy_score              # 정확도: 맞힌 개수 / 전체 개수

clf = LogisticRegression(
    max_iter=1000,        # 최적화 반복 상한. 딥러닝의 epoch 수와 비슷한 역할
    random_state=42,      # 시드 고정 (재현성)
).fit(Xtr_s, ytr)         # ← 학습. 표준화된 train 데이터만 사용한다

# clf.predict(...) 는 각 샘플의 예측 클래스(0/1/2)를 배열로 돌려준다
train_acc = accuracy_score(ytr, clf.predict(Xtr_s))   # 배운 데이터로 채점 (= 외운 걸 다시 물어봄)
test_acc = accuracy_score(yte, clf.predict(Xte_s))    # 처음 보는 데이터로 채점 (= 진짜 실력)

print(f"baseline train acc: {train_acc:.3f}")
print(f"baseline test  acc: {test_acc:.3f}   ← 이게 '일반화 성능'이다")
print(f"train-test 격차   : {train_acc - test_acc:+.3f}   ← 크게 벌어지면 과적합 신호")

### 결과 읽는 법

```
baseline train acc: 1.000
baseline test  acc: 0.981
train-test 격차   : +0.019
```

**`train 1.000` 을 보고 좋아하면 안 된다.** 배운 데이터를 다시 맞힌 것뿐이다 —
시험 문제를 미리 보고 푼 점수다. 봐야 할 숫자는 **`test 0.981`**, 그리고 **둘의 격차**다.

| 격차 | 해석 |
|---|---|
| 작다 (`+0.019`) | 건강하다. 외운 게 아니라 규칙을 배웠다 |
| 크다 (`+0.3` 이상) | **과적합(overfitting)** — train 만 잘하고 실전에서 무너진다 |

> 🧠 **딥러닝의 train loss ↓ / val loss ↑ 갈라지는 그래프**와 같은 이야기다.
> 여기서는 epoch 축이 없으니 곡선 대신 **두 숫자의 차이**로 본다.

## 4. 평가 — 정확도 하나로는 부족하다 (혼동행렬)

정확도 98%는 "54병 중 53병을 맞혔다"만 말해 준다. **어느 품종을 어느 품종으로
착각했는지**는 안 알려준다. 그걸 보는 게 **혼동행렬(confusion matrix)** 이다.

읽는 법은 간단하다:
- **행 = 실제 정답**, **열 = 모델의 예측**
- **대각선** = 맞힌 것, **대각선 밖** = 틀린 것

딥러닝에서 쓰던 그 혼동행렬과 완전히 같다.

In [ ]:
# confusion_matrix: 실제 vs 예측 교차표 / classification_report: 클래스별 지표 요약
from sklearn.metrics import classification_report, confusion_matrix

pred = clf.predict(Xte_s)              # test 데이터에 대한 예측 (0/1/2 배열)
cm = confusion_matrix(yte, pred)       # (3, 3) 행렬 — cm[i][j] = 실제 i 인데 j 로 예측한 개수
w = max(len(n) for n in names)         # 출력 정렬용: 가장 긴 클래스 이름 길이

print("혼동행렬 (행=실제, 열=예측):")
# 헤더 줄: 왼쪽 여백(w+2칸) 뒤에 클래스 이름들을 오른쪽 정렬로 나열
print(" " * (w + 2) + "  ".join(f"{n:>{w}}" for n in names))

for i, row in enumerate(cm):           # 행렬을 한 줄씩
    # 줄 맨 앞에 '실제 클래스' 이름을 찍고, 그 행의 숫자들을 정렬해 출력
    print(f"{names[i]:>{w}}  " + "  ".join(f"{v:>{w}}" for v in row))

print()
# classification_report: 클래스별 precision/recall/f1 을 한 번에 표로
print(classification_report(yte, pred, target_names=names, digits=2))

### 결과 읽는 법

```
혼동행렬 (행=실제, 열=예측):
         class_0  class_1  class_2
class_0       18        0        0
class_1        1       20        0
class_2        0        0       15
```

**`class_1` 행의 `1`** ← 이 한 칸이 핵심이다.
**실제로는 `class_1` 인데 모델이 `class_0` 이라고 답한 와인이 1병** 있었다는 뜻이다.

54병 중 딱 하나 틀렸고, **그 하나가 어느 쌍에서 났는지**까지 보인다.
정확도 `0.98` 만 봤다면 "1병 틀렸구나"에서 끝났을 정보다.

### precision / recall (딥러닝에서 쓰던 그것과 동일)

- **precision(정밀도)**: 모델이 "class_0!"이라 한 것 중 진짜 class_0 의 비율 → *헛다리 안 짚나*
- **recall(재현율)**: 진짜 class_0 중 모델이 찾아낸 비율 → *놓치지 않나*

`class_0` 의 precision 이 `0.95` 인 이유가 바로 위의 그 1병이다 —
class_0 이라고 답한 19병 중 1병이 사실 class_1 이었다.

## 5. 실험 반복은 손발이다 — 위임할 부분

여기서 모델을 바꿔 본다. **랜덤 포레스트(Random Forest)** 는 고전 ML 의 대표 주자다.

### 랜덤 포레스트를 딥러닝 감각으로 이해하기

- **결정 트리(decision tree)** 하나 = "알코올 > 13이면 왼쪽, 아니면 오른쪽..." 식의
  스무고개. 규칙을 계속 쪼개 내려가며 분류한다.
- 트리 하나는 잘 외워 버린다(과적합). 그래서 **데이터와 특성을 조금씩 다르게 준
  트리를 수백 개 만들어 다수결**을 시킨다. 그게 랜덤 포레스트다.
- **역전파도 경사하강법도 없다.** 미분 자체를 안 쓴다. 완전히 다른 계열의 알고리즘이다.

`n_estimators` = **트리를 몇 그루 심을지**. 딥러닝의 "층을 몇 개 쌓을까"에 해당하는,
가장 먼저 만져 보는 하이퍼파라미터다.

**이런 반복 실험이야말로 에이전트에게 맡길 일이다**(교안 3.1·3.7) —
사람은 *어떤 축을 실험할지*만 정한다.

In [ ]:
from sklearn.ensemble import RandomForestClassifier   # 결정 트리 여러 그루의 다수결

print(f"{'n_estimators':>12} | {'test acc':>8}")   # 표 헤더 (>12 = 폭 12로 오른쪽 정렬)
print("-" * 25)

for n in [10, 50, 100, 200]:                       # 트리 개수를 바꿔 가며 반복 = '스윕'
    rf = RandomForestClassifier(
        n_estimators=n,       # 트리 개수
        random_state=42,      # 시드 고정 (트리마다 랜덤 샘플링을 하므로 필수)
    ).fit(Xtr_s, ytr)         # 학습 (여기도 train 만)
    print(f"{n:>12} | {accuracy_score(yte, rf.predict(Xte_s)):>8.3f}")

print()
print("→ 나무를 늘려도 어느 지점부터는 안 오른다. '더 크게'가 답이 아닐 때가 많다.")

### 결과 읽는 법

```
n_estimators | test acc
          10 |    0.963
          50 |    1.000
         100 |    1.000
         200 |    1.000
```

**`50` 이후로는 트리를 4배 늘려도 그대로다.** 학습 시간만 4배 쓴 셈이다.

딥러닝에서 "파라미터를 늘리면 대체로 좋아진다"는 감각이 있다면, 고전 ML 에서는
**금방 천장에 닿는다**는 걸 기억해 두면 좋다. 데이터가 178개뿐이니 더 짜낼 정보가 없다.

> ⚠️ 그리고 `1.000` 은 **의심해야 할 숫자**다. 54병을 다 맞혔다는 뜻인데,
> 데이터가 이만큼 작으면 우연히 그럴 수 있다. 진짜로 확인하려면 교차검증
> (cross-validation)으로 여러 번 나눠 재 봐야 한다.

## 6. 🔥 함정 ① — 데이터 누수(leakage)를 점수로 확인한다

**이 절이 3교시 전체의 핵심이다.** 앞의 모든 절은 여기를 위한 준비였다.

### 실험 설계 — 일부러 '배울 게 없는 데이터'를 만든다

- **특성**: 완전 난수 2000개
- **정답**: 완전 난수 (동전 던지기)
- 둘 사이에 **아무 관계도 없다.**

그러니 어떤 천재적인 모델이라도 정답률은 **50%(찍기)** 여야 한다. 그게 정답이다.

### 두 가지 방식으로 돌려 본다

| | 순서 | 무슨 문제가 |
|---|---|---|
| ✗ 누수 | **전체 데이터**로 유용한 특성 20개 고르기 → 그 다음 분할 | 고를 때 **test 의 정답을 봤다** |
| ✓ 올바름 | 분할 → **train 만 보고** 특성 20개 고르기 | test 는 끝까지 안 봤다 |

**두 코드 모두 문법 오류가 없고 완벽하게 실행된다.** 차이는 `train_test_split` 이
몇 번째 줄에 있느냐뿐이다. 그래서 무섭다.

In [ ]:
from sklearn.feature_selection import SelectKBest, f_classif
# SelectKBest: '정답과 관계가 깊어 보이는' 특성 상위 k개만 남기는 도구
# f_classif  : 그 '관계 깊음'을 재는 통계 기준(ANOVA F값)

rng = np.random.default_rng(0)                  # 난수 생성기 (시드 0 고정)
X_noise = rng.normal(size=(200, 2000))          # 특성: 200샘플 × 2000개, 전부 정규분포 난수
y_noise = rng.integers(0, 2, size=200)          # 정답: 0 또는 1을 무작위로 → 신호 0


def evaluate(leaky: bool) -> float:
    # train_test_split 의 반환 순서는 항상 (X_train, X_test, y_train, y_test) 다.
    # 아래에서 a=X_train, b=X_test, c=y_train, d=y_test 로 받는다.
    if leaky:
        # ✗ 누수 경로 --------------------------------------------------
        # 전체 데이터(X_noise 전부 = test 될 부분까지)를 보고 특성 20개를 고른다.
        # 이 시점에 이미 'test 의 정답과 잘 맞는 특성'이 뽑혀 버린다.
        X_sel = SelectKBest(f_classif, k=20).fit_transform(X_noise, y_noise)   # ← 여기가 범인
        a, b, c, d = train_test_split(X_sel, y_noise, test_size=0.3, random_state=0)  # 고른 뒤에 분할
    else:
        # ✓ 올바른 경로 ------------------------------------------------
        a, b, c, d = train_test_split(X_noise, y_noise, test_size=0.3, random_state=0)  # 먼저 분할
        sel = SelectKBest(f_classif, k=20).fit(a, c)   # train(a, c) 만 보고 고른다
        a, b = sel.transform(a), sel.transform(b)      # 고른 기준으로 둘 다 변환만

    model = LogisticRegression(max_iter=1000).fit(a, c)   # train 으로 학습
    return accuracy_score(d, model.predict(b))            # test 로 채점


print("데이터: 특성도 정답도 난수 — 배울 신호가 전혀 없다")
print()
print(f"누수 있는 버전 (✗): test acc = {evaluate(leaky=True):.3f}   ← 신호가 없는데 이 점수")
print(f"올바른 버전   (✓): test acc = {evaluate(leaky=False):.3f}   ← 동전 던지기 수준 = 정직")

### 🔥 결과 — 없는 실력이 만들어졌다

```
누수 있는 버전 (✗): test acc = 0.750
올바른 버전   (✓): test acc = 0.450
```

데이터에는 **배울 게 0** 이다. 정직한 답은 `0.450`(찍기 수준)이 맞다.
그런데 누수 버전은 **0.750** 을 "달성"했다. **+0.300 만큼 없는 실력이 만들어진 것이다.**

### 왜 이런 일이 생기나

난수 특성 2000개 중에는, **순전히 우연으로** test 의 정답과 잘 맞아떨어지는 게 반드시
몇 개 있다. 전체 데이터를 보고 특성을 고르면 `SelectKBest` 가 **바로 그 우연들을 골라 준다.**
모델은 그 우연을 학습하고, 채점도 같은 우연으로 받는다. 그래서 점수가 오른다.

### 실무에서 이게 왜 재앙인가

- 개발 중: **"정확도 75% 모델 완성"** 이라고 보고된다
- 배포 후: 실제로는 50%. **서비스가 동전 던지기를 하고 있다**
- 원인 추적: 코드에 에러가 없으니 어디가 문제인지 안 보인다

> 🧠 **딥러닝에서도 정확히 같은 일이 일어난다.** 전체 데이터로 정규화 통계를 계산하거나,
> validation 을 포함해 특성 선택·하이퍼파라미터 탐색을 하거나, 시계열에서 미래 데이터를
> 섞어 넣는 경우. **프레임워크의 문제가 아니라 절차의 문제**다.

> 💡 **막는 법**: 전처리를 전부 `sklearn.pipeline.Pipeline` 안에 넣으면
> `fit` 이 train 에만 적용되는 것이 **구조적으로 보장**된다 — 사람이 순서를 기억할
> 필요가 없어진다. 교안 3.10에서 이 방식으로 다시 만들고, **누수를 자동으로 잡는
> 테스트**까지 붙인다.

## 7. 함정 ② — 정확도가 거짓말을 할 때

**불균형 데이터(imbalanced data)**: 찾아야 할 대상이 아주 드문 경우다.
불량품 검출, 희귀병 진단, 이상거래 탐지가 전부 여기에 해당한다.

이럴 때 **아무것도 학습하지 않고 "전부 정상"이라고만 찍는 모델**의 점수를 보자.

In [ ]:
from sklearn.dummy import DummyClassifier   # 학습을 안 하는 '가짜' 모델 (기준선 확인용)
from sklearn.metrics import f1_score, recall_score   # accuracy 로는 안 보이는 것을 보는 지표들

rng2 = np.random.default_rng(7)                    # 셀마다 독립 시드 → 실행 순서 무관
X_imb = rng2.normal(size=(1000, 5))                # 입력 1000개 (내용은 중요하지 않다)
y_imb = (rng2.random(1000) < 0.02).astype(int)     # 약 2%만 1(양성), 나머지는 0(음성)

Xi_tr, Xi_te, yi_tr, yi_te = train_test_split(
    X_imb, y_imb, test_size=0.3, random_state=0
)

dummy = DummyClassifier(
    strategy="most_frequent",   # 전략: 무조건 '가장 많은 클래스'로 찍는다 → 항상 0
).fit(Xi_tr, yi_tr)             # fit 하지만 사실 배우는 게 없다

dp = dummy.predict(Xi_te)       # 예측 결과: 전부 0

print(f"양성 비율: {y_imb.mean():.1%}")
print(f"  accuracy = {accuracy_score(yi_te, dp):.3f}")   # 전체 중 맞힌 비율
print(f"  recall   = {recall_score(yi_te, dp, zero_division=0):.3f}")  # 진짜 양성 중 찾아낸 비율
print(f"  f1       = {f1_score(yi_te, dp, zero_division=0):.3f}")      # precision·recall 조화평균

### 결과 — 98점짜리 무능한 모델

```
양성 비율: 2.6%
  accuracy = 0.983
  recall   = 0.000
  f1       = 0.000
```

**`accuracy 0.983`.** 아무것도 학습하지 않았는데 98점이다.
양성이 2.6%뿐이니 **전부 "정상"이라고만 찍어도 97% 이상은 자동으로 맞는다.**

그런데 **`recall 0.000`** — 정작 찾아야 할 것을 **하나도 못 잡았다.**
불량품 검출기라면 불량을 전부 통과시킨 것이고, 질병 진단이라면 환자를 전부 놓친 것이다.

### 그래서 지표는 문제에 따라 고른다

| 상황 | 봐야 할 지표 | 이유 |
|---|---|---|
| 놓치면 큰일 (암 진단, 불량 검출) | **recall** | 못 찾은 게 몇 개인가 |
| 잘못 잡으면 곤란 (스팸 분류) | **precision** | 헛다리 짚은 게 몇 개인가 |
| 둘 다 중요 | **f1** | 두 지표의 조화평균 |

> ⚠️ **에이전트에게 지표를 지정해 주지 않으면 대개 accuracy 를 보고한다.**
> "정확도 98% 나왔습니다"라는 보고를 그대로 믿으면 안 되는 이유다 —
> **무엇을 맞혀야 하는 문제인지는 사람이 정해 줘야 한다.**

## 정리 — 오늘 배운 것

### 파이프라인 (1~5절)
```
로드·탐색  →  분할  →  스케일링  →  학습  →  평가  →  반복
   ↑           ↑ 순서가 생명           ↑ baseline 먼저   ↑ 정확도만 보지 말 것
   무엇이 필요한지 정하는 단계
```

### 함정 두 개 (6~7절)

| 함정 | 실측 | 무엇이 문제인가 |
|---|---|---|
| **데이터 누수** | 신호 0인 데이터에서 `0.750` vs `0.450` | 없는 실력이 +0.300 만들어짐 |
| **잘못된 지표** | `accuracy 0.983` / `recall 0.000` | 아무것도 안 하는 모델이 98점 |

**두 경우 모두 코드는 완벽하게 실행된다.** 문법 오류가 아니라 **방법론 오류**이기 때문이다.
그래서 실행 결과만 봐서는 절대 알 수 없다.

### 딥러닝 하던 사람에게 남길 한 줄

고전 ML 은 모델이 단순한 대신 **절차가 실력**이다. 어떤 층을 쌓을지가 아니라
**언제 무엇을 보면 안 되는지**를 지키는 게 성능을 만든다.
그리고 그 절차는 **AI 가 대신 지켜 주지 않는다** — 시키지 않으면 안 한다.

> 다음: 교안 **3.10** 에서 이 파이프라인을 에이전트에게 맡기고,
> **누수를 자동으로 잡아내는 테스트 게이트**를 만들어 붙인다.
> "사람이 검증해야 한다"를 각오가 아니라 **자동 장치**로 바꾸는 단계다.